# 06. 이미지-텍스트 검색과 VLM식 증거 검색

VLM을 학습하기 전에도 이미지 설명, OCR 텍스트, 질문을 같은 검색 공간에 두는 연습이 필요합니다.
여기서는 TF-IDF로 단순 검색 베이스라인을 만들고 Recall@K를 계산합니다.


In [ ]:
from pathlib import Path
import json
import math
import random
import statistics
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except Exception as exc:
    plt = None
    print("matplotlib을 불러오지 못했습니다:", exc)

ROOT = Path.cwd()
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

random.seed(42)
np.random.seed(42)

def save_json(name, obj):
    path = ARTIFACTS / name
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    return path

def display_df(df, n=20):
    return df.head(n)


In [ ]:
items = pd.DataFrame([
    ("img001", "receipt total 42000 paid sensor", "가격이 표시된 영수증"),
    ("img002", "warning alarm red light emergency stop", "알람과 정지 안내"),
    ("img003", "manual robot arm calibration joint", "로봇 암 보정 매뉴얼"),
    ("img004", "meeting room schedule projector", "회의실 일정표"),
    ("img005", "invoice number a17 status paid", "청구서 번호와 결제 상태"),
], columns=["image_id", "evidence_text", "caption_ko"])

queries = pd.DataFrame([
    ("q1", "문서 번호 A17이 있는 이미지를 찾아줘", "img005"),
    ("q2", "알람이 울리는 위험 상황", "img002"),
    ("q3", "센서 가격이 있는 영수증", "img001"),
], columns=["query_id", "query", "target_image_id"])


In [ ]:
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity

    corpus = pd.concat([items["evidence_text"], queries["query"]], ignore_index=True)
    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4))
    X = vectorizer.fit_transform(corpus)
    item_vecs = X[:len(items)]
    query_vecs = X[len(items):]
    sims = cosine_similarity(query_vecs, item_vecs)

    ranked_rows = []
    for qi, qrow in queries.iterrows():
        order = np.argsort(-sims[qi])
        for rank, idx in enumerate(order[:3], start=1):
            ranked_rows.append({
                "query_id": qrow.query_id,
                "rank": rank,
                "image_id": items.iloc[idx].image_id,
                "score": float(sims[qi, idx]),
                "hit": items.iloc[idx].image_id == qrow.target_image_id,
            })
    ranking = pd.DataFrame(ranked_rows)
    recall_at_1 = ranking[ranking["rank"] == 1]["hit"].mean()
    save_json("retrieval_metrics.json", {"recall_at_1": float(recall_at_1)})
    print("Recall@1:", recall_at_1)
    ranking
except Exception as exc:
    print("scikit-learn 설치 후 실행하세요:", exc)


## 확장 과제

- OCR 텍스트와 이미지 캡션을 따로 색인하고 late fusion을 구현합니다.
- 실제 VLM 임베딩 모델을 사용할 경우 보고서 본문에는 회사명/조직명을 `xxx`로 마스킹합니다.
- 검색 실패 사례를 `텍스트 누락`, `동의어`, `숫자/코드`, `시각 단서 부족`으로 분류합니다.
